# Exercise 20.1: In-Memory RAG Retriever

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-20/01-in-memory-rag-retriever.ipynb)

- **Book Reference:** Chapter 20 (*Embeddings, Vector Search (RAG) and Fine-Tuning (LoRA)*, Section 20.1)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Implement pairwise and batch cosine similarity $\operatorname{sim}(\mathbf{q}, \mathbf{d}_i) = \frac{\mathbf{q} \cdot \mathbf{d}_i}{\|\mathbf{q}\| \|\mathbf{d}_i\|}$.
  2. Build an in-memory vector retriever that ranks document chunks and returns the top-$k$ matches.
  3. Verify the Chapter 20 worked example $\mathbf{q} = (1, 1)$, $\mathbf{d}_1 = (2, 2)$, $\mathbf{d}_2 = (1, -1)$.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## Mathematical Background

In **Retrieval-Augmented Generation (RAG)**, each document chunk is embedded as a vector $\mathbf{d}_i \in \mathbb{R}^d$, and the user's query is embedded as $\mathbf{q} \in \mathbb{R}^d$. Relevance is measured by **cosine similarity**:
$$\operatorname{sim}(\mathbf{q}, \mathbf{d}_i) = \frac{\mathbf{q} \cdot \mathbf{d}_i}{\lVert \mathbf{q} \rVert \lVert \mathbf{d}_i \rVert}$$

### Worked Example Check (Chapter 20.1)
With $\mathbf{q} = (1, 1)$, $\mathbf{d}_1 = (2, 2)$, $\mathbf{d}_2 = (1, -1)$:
- $\operatorname{sim}(\mathbf{q}, \mathbf{d}_1) = \frac{2 + 2}{\sqrt{2}\sqrt{8}} = 1.0$
- $\operatorname{sim}(\mathbf{q}, \mathbf{d}_2) = \frac{1 - 1}{\sqrt{2}\sqrt{2}} = 0.0$

So RAG retrieves $\mathbf{d}_1$ as the top-1 chunk.

In [ ]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Compute cosine similarity between two 1D vectors u and v.

    Args:
        u: 1D NumPy array of shape (d,).
        v: 1D NumPy array of shape (d,).

    Returns:
        Cosine similarity in [-1.0, 1.0] (0.0 if either vector has zero norm).
    """
    # TODO: Compute dot(u, v) / (||u|| * ||v||), guarding against zero norm
    raise NotImplementedError("Complete this TODO")


def cosine_similarity_batch(query: np.ndarray, docs: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between a query vector and a matrix of document vectors.

    Args:
        query: 1D array of shape (d,).
        docs: 2D array of shape (N, d).

    Returns:
        1D array of cosine similarities of shape (N,).
    """
    # TODO: Compute vectorised cosine similarities across all N rows of docs
    raise NotImplementedError("Complete this TODO")


def retrieve_top_k(
    query: np.ndarray,
    docs: np.ndarray,
    texts: list[str],
    k: int = 3,
) -> list[tuple[int, float, str]]:
    """Retrieve the top-k most similar document chunks to a query.

    Args:
        query: Query embedding of shape (d,).
        docs: Document embedding matrix of shape (N, d).
        texts: List of N document chunk strings.
        k: Number of top chunks to return.

    Returns:
        List of (doc_index, score, text) sorted by descending similarity score.
    """
    # TODO: Rank scores descending and return top k tuples (idx, float(score), texts[idx])
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- Use `np.linalg.norm(docs, axis=1)` for row-wise norms.
- `np.argsort(-scores)[:k]` gives the indices of the `k` largest scores in descending order.
- Always guard against division by zero when a vector norm is `0.0`.

</details>

In [ ]:
# 1. Worked example from Chapter 20.1
q_ex = np.array([1.0, 1.0])
d_ex = np.array([[2.0, 2.0], [1.0, -1.0]])
texts_ex = ["Chunk A: aligned", "Chunk B: orthogonal"]

assert np.isclose(cosine_similarity(q_ex, d_ex[0]), 1.0)
assert np.isclose(cosine_similarity(q_ex, d_ex[1]), 0.0)
top1 = retrieve_top_k(q_ex, d_ex, texts_ex, k=1)
assert top1[0][0] == 0 and np.isclose(top1[0][1], 1.0)

# 2. Exercise 20.1: q=(3,4), d=(6,8)
assert np.isclose(cosine_similarity(np.array([3.0, 4.0]), np.array([6.0, 8.0])), 1.0)

# 3. Synthetic 8-chunk knowledge base
kb_texts = [f"Policy chunk {i}" for i in range(8)]
kb_vecs = rng.normal(size=(8, 16))
# Make chunk 5 strongly aligned with query
query_vec = rng.normal(size=16)
kb_vecs[5] = 3.0 * query_vec + 0.05 * rng.normal(size=16)

results = retrieve_top_k(query_vec, kb_vecs, kb_texts, k=3)
assert len(results) == 3
assert results[0][0] == 5, f"Expected chunk 5 first, got {results[0][0]}"
assert results[0][1] >= results[1][1] >= results[2][1]
print("All checks passed! Top-3 results:", [(i, round(s, 4)) for i, s, _ in results])

## Think Deeper

1. Why is RAG preferred over fine-tuning when adding frequently changing company policies to an LLM?
2. What happens to cosine similarity if every document vector is scaled by $10$? Does the top-$k$ ranking change?